# Skin Image CNN — corrected training notebook

This notebook trains the supplied `ISICCNN` architecture using real images in `data/` (or `DATA_DIR`). The original notebook generated random placeholder images and derived labels from brightness. These labels are **not cancer diagnoses**. Missing images are not fabricated.


In [ ]:
from pathlib import Path
import os, json, joblib
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

DATA_DIR = Path(os.environ.get('DATA_DIR', 'data')).expanduser().resolve()
OUTPUT_DIR = Path('outputs').resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Image folder:', DATA_DIR, '| Device:', DEVICE)


In [ ]:
# Discover real images only — do not generate random placeholders.
image_paths = sorted([p for ext in ('*.jpg', '*.jpeg', '*.png') for p in DATA_DIR.glob(ext)])
if len(image_paths) < 2:
    raise RuntimeError(f'Found {len(image_paths)} images in {DATA_DIR}. Add at least 2 real JPG/PNG images.')
labels, valid_paths = [], []
for path in image_paths:
    try:
        with Image.open(path) as im:
            gray = np.asarray(im.convert('L'), dtype=np.float32)
        labels.append(int(gray.mean() > 127))
        valid_paths.append(path)
    except Exception as exc:
        print(f'Skipping unreadable image {path.name}: {exc}')
image_paths = valid_paths
if len(image_paths) < 2 or len(set(labels)) < 2:
    raise RuntimeError('Need at least two readable images and both brightness classes.')
print(f'{len(image_paths)} images loaded; class counts: {np.bincount(labels, minlength=2)}')


In [ ]:
class ISICDataset(Dataset):
    def __init__(self, paths, targets, transform=None): self.paths, self.targets, self.transform = paths, targets, transform
    def __len__(self): return len(self.paths)
    def __getitem__(self, idx):
        with Image.open(self.paths[idx]) as im: image = im.convert('RGB')
        if self.transform: image = self.transform(image)
        return image, torch.tensor(self.targets[idx], dtype=torch.long)

train_transform = transforms.Compose([transforms.Resize((128,128)), transforms.RandomHorizontalFlip(), transforms.RandomRotation(15), transforms.ToTensor(), transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
val_transform = transforms.Compose([transforms.Resize((128,128)), transforms.ToTensor(), transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
train_paths, val_paths, train_labels, val_labels = train_test_split(image_paths, labels, test_size=0.2, random_state=42, stratify=labels)
train_dataset = ISICDataset(train_paths, train_labels, train_transform)
val_dataset = ISICDataset(val_paths, val_labels, val_transform)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=0)
print('Train:', len(train_dataset), '| Validation:', len(val_dataset))


In [ ]:
class ISICCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(nn.Conv2d(3,16,3,padding=1),nn.BatchNorm2d(16),nn.ReLU(),nn.MaxPool2d(2,2),nn.Conv2d(16,32,3,padding=1),nn.BatchNorm2d(32),nn.ReLU(),nn.MaxPool2d(2,2),nn.Conv2d(32,64,3,padding=1),nn.BatchNorm2d(64),nn.ReLU(),nn.MaxPool2d(2,2))
        self.classifier = nn.Sequential(nn.Linear(64*16*16,128),nn.ReLU(),nn.Dropout(0.5),nn.Linear(128,2))
    def forward(self,x): return self.classifier(self.features(x).flatten(1))
model = ISICCNN().to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [ ]:
EPOCHS = 5
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
for epoch in range(EPOCHS):
    model.train(); tl = tc = tt = 0
    for images, targets in train_loader:
        images, targets = images.to(DEVICE), targets.to(DEVICE)
        optimizer.zero_grad(set_to_none=True); outputs = model(images); loss = criterion(outputs, targets)
        loss.backward(); optimizer.step(); tl += loss.item()*targets.size(0)
        tc += (outputs.argmax(1)==targets).sum().item(); tt += targets.size(0)
    model.eval(); vl = vc = vt = 0
    with torch.inference_mode():
        for images, targets in val_loader:
            images, targets = images.to(DEVICE), targets.to(DEVICE); outputs = model(images); loss = criterion(outputs, targets)
            vl += loss.item()*targets.size(0); vc += (outputs.argmax(1)==targets).sum().item(); vt += targets.size(0)
    history['train_loss'].append(tl/max(tt,1)); history['train_acc'].append(tc/max(tt,1))
    history['val_loss'].append(vl/max(vt,1)); history['val_acc'].append(vc/max(vt,1))
    print(f"Epoch {epoch+1}/{EPOCHS}: train loss={history['train_loss'][-1]:.4f}, train acc={history['train_acc'][-1]:.3f}, val loss={history['val_loss'][-1]:.4f}, val acc={history['val_acc'][-1]:.3f}")


In [ ]:
fig, axes = plt.subplots(1,2,figsize=(11,4))
axes[0].plot(history['train_loss'],label='Train'); axes[0].plot(history['val_loss'],label='Validation'); axes[0].set_title('Loss'); axes[0].legend()
axes[1].plot(history['train_acc'],label='Train'); axes[1].plot(history['val_acc'],label='Validation'); axes[1].set_title('Accuracy'); axes[1].legend()
plt.tight_layout(); plt.show()
model.eval(); preds=[]; actual=[]
with torch.inference_mode():
    for images, targets in val_loader:
        preds.extend(model(images.to(DEVICE)).argmax(1).cpu().tolist()); actual.extend(targets.tolist())
print(classification_report(actual,preds,labels=[0,1],target_names=['Low brightness (0)','High brightness (1)'],zero_division=0))
print('Confusion matrix:\n',confusion_matrix(actual,preds,labels=[0,1]))


In [ ]:
# Output file format is compatible with cancer_detection_fixed.py
payload = {'state_dict': {k:v.detach().cpu() for k,v in model.state_dict().items()}, 'model_class':'ISICCNN', 'input_size':(128,128), 'device':'cpu', 'label_meaning':'brightness-derived demo labels; not cancer diagnoses'}
joblib.dump(payload, OUTPUT_DIR/'selected_cnn_model.joblib')
torch.save(payload['state_dict'], OUTPUT_DIR/'selected_cnn_model.pth')
with open(OUTPUT_DIR/'training_history.json','w',encoding='utf-8') as f: json.dump(history,f,indent=2)
print('Saved model and history in', OUTPUT_DIR)
